In [2]:
import os
import threading
import warnings

import numpy as np
import pywt
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal

import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure

warnings.filterwarnings("ignore")

### Класс Wavelet Declicker

Основной алгоритм: загрузка, детекция, ремонт, сохранение и визуализация (опционально)

Аудиофайл читается через `soundfile`, сигнал хранится в виде массива `float64`. На этом этапе определяется максимально допустимый уровень DWT, чтобы разложение было корректным для данной длины сигнала и выбранного вейвлета.
Сигнал раскладывается на `cA_L` и `cD_j`, чтоб занулить аппроксимацию и оставить для детекции только выбранный диапазон полос из списка `cD`. Далее вычисляются модуль сигнала, медиана и MAD-оценка sigma:
```
threshold = median + Detect MAD k * sigma
```
Все превышения этого порога считаются кандидатами в клики.\
Т.к. клик - это событие, превышающее по длине один сэмпл, то соседние выходы за порог объединяются в кластеры, и вычисляется центр каждого события.\
Результатом этапа детекции становится список временных индексов, в которых были обнаружены клики.

На этапе ремонта сигнал снова раскладывается для обработки выбранных полос.\
На уровне `j` один коэффициент соответствует примерно `2^j` сэмплам. Поэтому центр события переводится в индекс коэффициента:
```
k0 = n // 2^j
```
В контекстном окне (задаваемом пользователем) вычисляются медиана `m`, `sigma` через MAD и применяется тримминг - `sigma trim q`. В центре события измеряется:
```
out_strength = max(|coeff - m|) / sigma
```
Это определяет, насколько сильным является выброс относительно локального фона.\
Значение `k_auto` выбирается между `k_min` и `k_max`: слабые выбросы - `k` ближе к `k_max` (мягкая обработка), сильные выбросы - `k` ближе к `k_min` (жёсткая обработка). Если коэффициенты выходят за пределы, они ограничиваются Параметр `Reaction` управляет скоростью перехода.

После обработки всех событий сигнал восстанавливается.

Таким образом, алгоритм не затрагивает сигнал вне событий, не влияет на полосы вне выбранного диапазона, автоматически подстраивает силу подавления под амплитуду клика. Программа выполняет вейвлет-клиппинг коэффициентов в выбранном диапазоне, событийно и робастно (выявляет выбросы).

In [ ]:
class WaveletDeclicker:
    def __init__(self):
        self.audio_data = None
        self.sample_rate = None
        self.is_stereo = False
        self.stereo_channels = None
        self.cleaned_audio = None
        self.cleaned_stereo = None
        self.last_click_centers = []
        self.last_click_count = 0

        self.wavelet_families = {
            "Daubechies": ["db1", "db2", "db3", "db4", "db5", "db6", "db7", "db8", "db9", "db10"],
            "Symlets": ["sym2", "sym3", "sym4", "sym5", "sym6", "sym7", "sym8"],
            "Coiflets": ["coif1", "coif2", "coif3", "coif4", "coif5"],
            "Biorthogonal": [
                "bior1.1", "bior1.3", "bior1.5", "bior2.2", "bior2.4", "bior2.6",
                "bior2.8", "bior3.1", "bior3.3", "bior3.5", "bior3.7", "bior3.9"
            ],
            "Reverse Biorthogonal": [
                "rbio1.1", "rbio1.3", "rbio1.5", "rbio2.2", "rbio2.4",
                "rbio2.6", "rbio2.8", "rbio3.1", "rbio3.3", "rbio3.5",
                "rbio3.7", "rbio3.9"
            ],
            "Haar": ["haar"]
        }

        self.default_params = {
            "wavelet": "db4",
            "border_mode": "symmetric",
            "level": 10,
            "cycle_spins": 16,  # 0/8/16

            # Band range: cD1=HF ... cDlevel=LF
            "band_high": 1,
            "band_low": 10,

            # Detector
            "detect_mad_k": 10.0,

            # Event geometry
            "event_ms": 8.0,
            "context_ms": 30.0,

            # Robust sigma
            "sigma_trim_q": 0.90,
            "min_sigma": 1e-6,

            # Auto suppression
            "reaction": 1.0,
            "k_min": 1.0,
            "k_max": 5.0,
        }

    @staticmethod
    def _as_int(x, default):
        try:
            return int(x)
        except Exception:
            return int(default)

    @staticmethod
    def _as_float(x, default):
        try:
            return float(x)
        except Exception:
            return float(default)

    def _require_loaded(self):
        if self.sample_rate is None or self.audio_data is None:
            raise ValueError("Сначала загрузите аудио.")

    @staticmethod
    def _safe_level_for_signal(x_len, wavelet, requested_level):
        try:
            w = pywt.Wavelet(wavelet)
            max_level = pywt.dwt_max_level(data_len=int(x_len), filter_len=w.dec_len)
            return int(min(max(int(requested_level), 1), max_level))
        except Exception:
            return int(max(int(requested_level), 1))

    @staticmethod
    def _mad_sigma(x):
        x = np.asarray(x, dtype=np.float64)
        if x.size == 0:
            return 0.0
        med = np.median(x)
        mad = np.median(np.abs(x - med))
        if mad <= 0:
            return float(np.std(x) + 1e-12)
        return float(mad / 0.6745)

    @staticmethod
    def calculate_snr(original, cleaned):
        original = np.asarray(original, dtype=np.float64)
        cleaned = np.asarray(cleaned, dtype=np.float64)
        noise = original - cleaned
        sp = float(np.sum(original * original))
        npow = float(np.sum(noise * noise))
        if npow <= 1e-24:
            return float("inf")
        if sp <= 1e-24:
            return float("-inf")
        return float(10.0 * np.log10(sp / npow))

    # Bands (Hz)
    def approx_hz(self, L):
        """cA_L approx band: 0 .. Fs/2^(L+1)"""
        fs = float(self.sample_rate) if self.sample_rate else 0.0
        f_high = fs / (2.0 ** (L + 1))
        return 0.0, float(f_high)

    def detail_hz(self, j):
        """cD_j band approx: Fs/2^(j+1) .. Fs/2^j"""
        fs = float(self.sample_rate) if self.sample_rate else 0.0
        f_high = fs / (2.0 ** j)
        f_low = fs / (2.0 ** (j + 1))
        return float(f_low), float(f_high)

    def _normalize_band_range(self, level):
        bh = self._as_int(self.params.get("band_high", 1), 1)
        bl = self._as_int(self.params.get("band_low", level), level)
        bh = max(1, min(bh, level))
        bl = max(1, min(bl, level))
        if bh > bl:
            bh, bl = bl, bh
        self.params["band_high"] = bh
        self.params["band_low"] = bl
        return bh, bl

    def _selected_bands(self, level):
        bh, bl = self._normalize_band_range(level)
        return list(range(bh, bl + 1))

    def _trimmed_sigma(self, stats, m, trim_q, min_sigma):
        stats = np.asarray(stats, dtype=np.float64)
        if stats.size < 16:
            return max(float(self._mad_sigma(stats)), float(min_sigma))

        dev = np.abs(stats - m)
        if 0.5 < trim_q < 1.0 and dev.size >= 32:
            q = float(np.quantile(dev, trim_q))
            base = stats[dev <= q]
            sigma = self._mad_sigma(base) if base.size >= 16 else self._mad_sigma(stats)
        else:
            sigma = self._mad_sigma(stats)

        return max(float(sigma), float(min_sigma))
    # =======
    #   IO
    # =======
    def load_audio(self, filepath):
        try:
            audio, sr = sf.read(filepath, always_2d=False)
            self.sample_rate = int(sr)
            audio = np.asarray(audio, dtype=np.float64)

            self.is_stereo = False
            self.stereo_channels = None

            if audio.ndim > 1 and audio.shape[1] >= 2:
                self.is_stereo = True
                stereo = audio[:, :2]
                self.stereo_channels = stereo.T  # (2, N)
                self.audio_data = 0.5 * (stereo[:, 0] + stereo[:, 1])
            elif audio.ndim > 1:
                self.audio_data = audio[:, 0]
            else:
                self.audio_data = audio

            self.cleaned_audio = None
            self.cleaned_stereo = None
            return True
        except Exception as e:
            print(f"Ошибка загрузки: {e}")
            return False

    def save_cleaned_audio(self, filepath):
        if self.cleaned_audio is None and self.cleaned_stereo is None:
            raise ValueError("Нет очищенных данных для сохранения!")
        try:
            if self.is_stereo and (self.cleaned_stereo is not None):
                sf.write(filepath, self.cleaned_stereo, int(self.sample_rate))
            else:
                sf.write(filepath, self.cleaned_audio, int(self.sample_rate))
            return True
        except Exception as e:
            print(f"Ошибка сохранения: {e}")
            return False

    # =============
    #   DETECTOR
    # =============

    def _reconstruct_from_selected_details(self, x, wavelet, border_mode, level, selected_js):
        n = len(x)
        coeffs = pywt.wavedec(x, wavelet, level=level, mode=border_mode)
        out = [np.zeros_like(coeffs[0])]  # drop approximation for detection

        selected_js = set(selected_js)
        for idx in range(1, len(coeffs)):
            j = (len(coeffs) - 1) - (idx - 1)  # idx=len-1 -> j=1
            out.append(coeffs[idx] if (j in selected_js) else np.zeros_like(coeffs[idx]))

        y = pywt.waverec(out, wavelet, mode=border_mode)[:n]
        return y

    def detect_clicks(self, x_det):
        sr = int(self.sample_rate)
        wavelet = str(self.params.get("wavelet", "db4"))
        border_mode = str(self.params.get("border_mode", "symmetric"))
        level = int(self.params.get("level", 10))

        detect_mad_k = float(self.params.get("detect_mad_k", 10.0))
        event_ms = float(self.params.get("event_ms", 8.0))
        cluster_ms = max(1.0, event_ms)

        selected_js = self._selected_bands(level)
        if not selected_js:
            return []

        bp = self._reconstruct_from_selected_details(x_det, wavelet, border_mode, level, selected_js)

        a = np.abs(bp)
        med = float(np.median(a))
        sigma = self._mad_sigma(a)
        if sigma <= 1e-12:
            return []

        thr = med + detect_mad_k * sigma
        hits = np.where(a > thr)[0]
        if hits.size == 0:
            return []

        cluster = max(1, int(sr * (cluster_ms / 1000.0)))

        centers = []
        start = int(hits[0])
        prev = int(hits[0])
        for p in hits[1:]:
            p = int(p)
            if (p - prev) <= cluster:
                prev = p
            else:
                centers.append((start + prev) // 2)
                start = p
                prev = p
        centers.append((start + prev) // 2)
        return centers

    # ==========
    #   REPAIR
    # ==========

    def _auto_k_from_outlier(self, outlier_strength):
        reaction = float(self.params.get("reaction", 1.0))
        reaction = float(np.clip(reaction, 0.0, 1.0))

        k_min = float(self.params.get("k_min", 1.0))
        k_max = float(self.params.get("k_max", 5.0))
        if k_max < k_min:
            k_min, k_max = k_max, k_min

        s = max(0.0, float(outlier_strength))

        tau = 10.0 - 8.0 * reaction  # reaction=1 -> ~2 ; reaction=0 -> ~10
        tau = float(np.clip(tau, 1.5, 12.0))

        g = 1.0 - np.exp(-s / tau)  # 0..1
        k = k_max - (k_max - k_min) * g
        return float(np.clip(k, k_min, k_max))

    def _repair_band_around_events(self, d, j, click_centers):
        sr = int(self.sample_rate)

        event_ms = float(self.params.get("event_ms", 8.0))
        context_ms = float(self.params.get("context_ms", 30.0))

        trim_q = float(self.params.get("sigma_trim_q", 0.90))
        min_sigma = float(self.params.get("min_sigma", 1e-6))

        d = np.asarray(d, dtype=np.float64)
        ncoeff = len(d)
        if ncoeff < 64:
            return d

        scale = 2 ** j

        event_half_samp = max(1, int(sr * (event_ms / 1000.0) / 2.0))
        context_half_samp = max(event_half_samp + 1, int(sr * (context_ms / 1000.0) / 2.0))

        rj = int(np.ceil(event_half_samp / scale))
        wj = int(np.ceil(context_half_samp / scale))

        rj = max(rj, 8)
        wj = max(wj, rj + 12)

        for n in click_centers:
            n = int(n)
            k0 = n // scale
            if k0 < 0 or k0 >= ncoeff:
                continue

            lo = max(0, k0 - wj)
            hi = min(ncoeff, k0 + wj + 1)

            clo = max(0, k0 - rj)
            chi = min(ncoeff, k0 + rj + 1)

            left = d[lo:clo] if clo > lo else np.array([], dtype=np.float64)
            right = d[chi:hi] if chi < hi else np.array([], dtype=np.float64)
            stats = np.concatenate([left, right]) if (left.size + right.size) > 0 else d[lo:hi]
            if stats.size < 32:
                continue

            m = float(np.median(stats))
            sigma = self._trimmed_sigma(stats, m, trim_q, min_sigma)

            center_seg = d[clo:chi]
            if center_seg.size == 0:
                continue

            peak_dev = float(np.max(np.abs(center_seg - m)))
            out_strength = peak_dev / float(sigma + 1e-12)

            k_auto = self._auto_k_from_outlier(out_strength)
            T = float(k_auto * sigma)

            margin = max(2, int(0.15 * rj))
            seg_lo = max(0, clo - margin)
            seg_hi = min(ncoeff, chi + margin)

            seg = d[seg_lo:seg_hi]
            delta = seg - m
            over = np.abs(delta) > T
            if np.any(over):
                seg2 = seg.copy()
                seg2[over] = m + np.clip(delta[over], -T, +T)
                d[seg_lo:seg_hi] = seg2

        return d

    def _process_channel_event_multiband(self, x, click_centers):
        x = np.asarray(x, dtype=np.float64)
        n = x.size
        if n < 256 or len(click_centers) == 0:
            return x.copy()

        wavelet = str(self.params.get("wavelet", "db4"))
        border_mode = str(self.params.get("border_mode", "symmetric"))
        level = int(self.params.get("level", 10))

        c_orig = pywt.wavedec(x, wavelet, level=level, mode=border_mode)
        c_work = [ci.copy() for ci in c_orig]

        selected_js = set(self._selected_bands(level))

        for j in selected_js:
            c_work[-j] = self._repair_band_around_events(c_work[-j], j, click_centers)

        for j in range(1, level + 1):
            if j not in selected_js:
                c_work[-j] = c_orig[-j]

        # approximation unchanged
        c_work[0] = c_orig[0]

        y = pywt.waverec(c_work, wavelet, mode=border_mode)[:n]
        return y

    # ===========================
    #   CYCLE SPINNING (stereo)
    # ===========================

    def _cycle_spinning_stereo(self, L, R):
        S = int(self.params.get("cycle_spins", 0))
        S = max(0, S)

        if S == 0:
            det = np.maximum(np.abs(L), np.abs(R))
            clicks = self.detect_clicks(det)
            L2 = self._process_channel_event_multiband(L, clicks)
            R2 = self._process_channel_event_multiband(R, clicks)
            return L2, R2, len(clicks)

        n = len(L)
        accL = np.zeros(n, dtype=np.float64)
        accR = np.zeros(n, dtype=np.float64)
        clicks_sum = 0

        for s in range(S):
            Ls = np.roll(L, s)
            Rs = np.roll(R, s)

            det = np.maximum(np.abs(Ls), np.abs(Rs))
            clicks = self.detect_clicks(det)
            clicks_sum += len(clicks)

            Lp = self._process_channel_event_multiband(Ls, clicks)
            Rp = self._process_channel_event_multiband(Rs, clicks)

            accL += np.roll(Lp, -s)
            accR += np.roll(Rp, -s)

        return accL / float(S), accR / float(S), int(round(clicks_sum / float(S)))

    # =============
    #   MAIN API
    # =============

    def apply_declicking(self, params=None):
        self._require_loaded()

        self.params = dict(self.default_params)
        if isinstance(params, dict):
            self.params.update(params)

        x = np.asarray(self.audio_data, dtype=np.float64)
        n = len(x)

        wavelet = str(self.params.get("wavelet", "db4"))
        level_req = self._as_int(self.params.get("level", 10), 10)
        level = self._safe_level_for_signal(n, wavelet, level_req)
        self.params["level"] = level

        # clamp band range to actual level
        if self._as_int(self.params.get("band_low", level), level) > level:
            self.params["band_low"] = level
        self._normalize_band_range(level)

        # сбрасываем старую детекцию
        self.last_click_centers = []
        self.last_click_count = 0

        if not self.is_stereo:
            clicks = self.detect_clicks(np.abs(x))

            self.last_click_centers = list(clicks)
            self.last_click_count = len(clicks)

            y = self._process_channel_event_multiband(x, clicks)

            self.cleaned_audio = y
            self.cleaned_stereo = None

            return y, len(clicks)

        # stereo
        L = np.asarray(self.stereo_channels[0], dtype=np.float64)
        R = np.asarray(self.stereo_channels[1], dtype=np.float64)

        det_base = np.maximum(np.abs(L), np.abs(R))
        base_clicks = self.detect_clicks(det_base)

        self.last_click_centers = list(base_clicks)
        self.last_click_count = len(base_clicks)

        L2, R2, nclick = self._cycle_spinning_stereo(L, R)

        self.cleaned_stereo = np.vstack([L2, R2]).T
        self.cleaned_audio = 0.5 * (L2 + R2)

        return self.cleaned_audio, int(nclick)

    # visualization
    def visualize_results(self):
        if self.audio_data is None or self.cleaned_audio is None:
            raise ValueError("Нет данных для визуализации!")

        fig, axes = plt.subplots(3, 1, figsize=(14, 11))

        x = np.asarray(self.audio_data, dtype=np.float64)
        y = np.asarray(self.cleaned_audio, dtype=np.float64)
        sr = int(self.sample_rate)
        dur = len(x) / float(sr)

        t = np.arange(len(x)) / float(sr)

        axes[0].plot(t, x, linewidth=0.5, alpha=0.85)
        axes[0].set_title("Оригинал (полностью, mono view)")
        axes[0].grid(True, alpha=0.3)
        axes[0].set_xlim(0, dur)

        axes[1].plot(t, y, linewidth=0.5, alpha=0.85)
        axes[1].set_title("Очищенный (полностью, mono view)")
        axes[1].grid(True, alpha=0.3)
        axes[1].set_xlim(0, dur)

        diff = x - y
        f, tt, Sxx = signal.spectrogram(diff, sr, nperseg=256, noverlap=128)

        if tt.size > 0:
            tt = tt - tt[0]

        axes[2].pcolormesh(tt, f, 10 * np.log10(Sxx + 1e-10), shading="gouraud", cmap="hot")
        axes[2].set_title("Спектрограмма разности (orig - clean)")
        axes[2].set_xlabel("Время (сек)")
        axes[2].set_ylabel("Частота")
        axes[2].set_yscale("log")
        axes[2].set_ylim(20, sr / 2)
        axes[2].set_xlim(0, dur)

        yticks = [100, 1000, 5000, 10000, 20000]
        yticks = [v for v in yticks if 20 <= v <= (sr / 2)]
        axes[2].set_yticks(yticks)
        ylabels = []
        for v in yticks:
            if v >= 1000:
                ylabels.append(f"{int(v/1000)}k")
            else:
                ylabels.append(f"{int(v)}")
        axes[2].set_yticklabels(ylabels)

        max_sec = int(np.ceil(dur))
        xticks = list(range(0, max_sec + 1))
        axes[0].set_xticks(xticks)
        axes[1].set_xticks(xticks)
        axes[2].set_xticks(xticks)

        plt.tight_layout()
        return fig
    
    def visualize_click_segment(self, click_index=0, window_ms=120.0):
        if self.audio_data is None or self.cleaned_audio is None:
            raise ValueError("Нет данных для визуализации!")

        if not self.last_click_centers:
            raise ValueError("Нет обнаруженных кликов для отображения!")

        click_index = int(np.clip(click_index, 0, len(self.last_click_centers) - 1))
        center = int(self.last_click_centers[click_index])

        x = np.asarray(self.audio_data, dtype=np.float64)
        y = np.asarray(self.cleaned_audio, dtype=np.float64)
        sr = int(self.sample_rate)

        half = int(sr * (float(window_ms) / 1000.0) / 2.0)
        half = max(128, half)

        start = max(0, center - half)
        end = min(len(x), center + half)

        if end <= start:
            raise ValueError("Некорректный фрагмент для визуализации.")

        x_seg = x[start:end]
        y_seg = y[start:end]
        diff_seg = x_seg - y_seg

        t = np.arange(start, end) / float(sr)
        click_time = center / float(sr)

        fig, axes = plt.subplots(4, 1, figsize=(13, 10), sharex=False)

        axes[0].plot(t, x_seg, linewidth=0.8)
        axes[0].axvline(click_time, linestyle="--", linewidth=1.0)
        axes[0].set_title(
            f"Клик {click_index + 1}/{len(self.last_click_centers)} — оригинал, t = {click_time:.4f} сек"
        )
        axes[0].set_ylabel("Амплитуда")
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(t, y_seg, linewidth=0.8)
        axes[1].axvline(click_time, linestyle="--", linewidth=1.0)
        axes[1].set_title("Очищенный сигнал")
        axes[1].set_ylabel("Амплитуда")
        axes[1].grid(True, alpha=0.3)

        axes[2].plot(t, diff_seg, linewidth=0.8)
        axes[2].axvline(click_time, linestyle="--", linewidth=1.0)
        axes[2].set_title("Разность: original - cleaned")
        axes[2].set_ylabel("Амплитуда")
        axes[2].grid(True, alpha=0.3)

        nperseg = min(256, max(64, len(diff_seg) // 4))
        noverlap = nperseg // 2

        if len(diff_seg) >= nperseg:
            f, tt, Sxx = signal.spectrogram(
                diff_seg,
                sr,
                nperseg=nperseg,
                noverlap=noverlap
            )

            tt = tt + start / float(sr)

            axes[3].pcolormesh(
                tt,
                f,
                10 * np.log10(Sxx + 1e-10),
                shading="gouraud",
                cmap="hot"
            )

            axes[3].axvline(click_time, linestyle="--", linewidth=1.0)
            axes[3].set_yscale("log")
            axes[3].set_ylim(20, sr / 2)

            yticks = [100, 1000, 5000, 10000, 20000]
            yticks = [v for v in yticks if 20 <= v <= sr / 2]
            axes[3].set_yticks(yticks)

            ylabels = []
            for v in yticks:
                if v >= 1000:
                    ylabels.append(f"{int(v / 1000)}k")
                else:
                    ylabels.append(f"{int(v)}")
            axes[3].set_yticklabels(ylabels)

            axes[3].set_title("Спектрограмма разности вокруг клика")
            axes[3].set_ylabel("Частота")
            axes[3].set_xlabel("Время (сек)")
        else:
            axes[3].text(
                0.5,
                0.5,
                "Фрагмент слишком короткий для спектрограммы",
                ha="center",
                va="center",
                transform=axes[3].transAxes
            )
            axes[3].set_axis_off()

        for ax in axes[:3]:
            ax.set_xlim(t[0], t[-1])

        plt.tight_layout()
        return fig

### Класс Declicker GUI

Обёртка для ядра программы, в данной версии выполняет функцию временного интерфейса и создана для удобства проведения экспериментов.

In [ ]:
class DeclickerGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Wavelet Declicker (DWT)")
        self.root.geometry("1440x720")

        self.declicker = WaveletDeclicker()
        self.audio_loaded = False
        self.processing = False
        self.loaded_path = None

        self._setup_ui()

    def _setup_ui(self):
        toolbar = ttk.Frame(self.root)
        toolbar.pack(side=tk.TOP, fill=tk.X, padx=5, pady=5)

        ttk.Button(toolbar, text="Загрузить аудио", command=self.load_audio).pack(side=tk.LEFT, padx=2)
        ttk.Button(toolbar, text="Обработать", command=self.process_audio).pack(side=tk.LEFT, padx=2)
        ttk.Button(toolbar, text="Сохранить результат", command=self.save_result).pack(side=tk.LEFT, padx=2)
        ttk.Button(toolbar, text="Визуализация", command=self.visualize).pack(side=tk.LEFT, padx=2)

        ttk.Separator(self.root, orient=tk.HORIZONTAL).pack(fill=tk.X, padx=5, pady=5)

        main = ttk.PanedWindow(self.root, orient=tk.HORIZONTAL)
        main.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        left = ttk.LabelFrame(main, text="Параметры", padding=10)
        main.add(left, weight=1)

        right = ttk.Frame(main)
        main.add(right, weight=2)

        r = 0

        ttk.Label(left, text="Family:").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.wavelet_family_var = tk.StringVar(value="Daubechies")
        ttk.Combobox(left, textvariable=self.wavelet_family_var,
                     values=list(self.declicker.wavelet_families.keys()),
                     state="readonly", width=18).grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)

        ttk.Label(left, text="Wavelet:").grid(row=r, column=2, sticky=tk.W, pady=3)
        self.wavelet_var = tk.StringVar(value=self.declicker.default_params["wavelet"])
        self.wavelet_combo = ttk.Combobox(left, textvariable=self.wavelet_var, state="readonly", width=12)
        self.wavelet_combo.grid(row=r, column=3, sticky=tk.W, padx=5, pady=3)
        self._update_wavelet_list()
        self.wavelet_family_var.trace_add("write", self._update_wavelet_list)
        r += 1

        ttk.Label(left, text="Border mode:").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.border_mode_var = tk.StringVar(value=self.declicker.default_params["border_mode"])
        ttk.Combobox(left, textvariable=self.border_mode_var,
                     values=["symmetric", "reflect", "periodization", "per", "zero"],
                     state="readonly", width=16).grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)

        ttk.Label(left, text="DWT level:").grid(row=r, column=2, sticky=tk.W, pady=3)
        self.level_var = tk.IntVar(value=int(self.declicker.default_params["level"]))
        self.level_spin = ttk.Spinbox(left, from_=1, to=15, textvariable=self.level_var, width=8)
        self.level_spin.grid(row=r, column=3, sticky=tk.W, padx=5, pady=3)
        r += 1

        ttk.Label(left, text="Cycle spins:").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.cycle_spins_var = tk.IntVar(value=int(self.declicker.default_params["cycle_spins"]))
        ttk.Combobox(left, textvariable=self.cycle_spins_var, values=[0, 8, 16], state="readonly", width=8)\
            .grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)
        r += 1

        ttk.Separator(left, orient=tk.HORIZONTAL).grid(row=r, column=0, columnspan=4, sticky=tk.EW, pady=8)
        r += 1

        ttk.Label(left, text="Band high (HF, cD1..):").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.band_high_var = tk.IntVar(value=int(self.declicker.default_params["band_high"]))
        self.band_high_spin = ttk.Spinbox(left, from_=1, to=15, textvariable=self.band_high_var, width=8)
        self.band_high_spin.grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)

        ttk.Label(left, text="Band low (LF, cDlevel):").grid(row=r, column=2, sticky=tk.W, pady=3)
        self.band_low_var = tk.IntVar(value=int(self.declicker.default_params["band_low"]))
        self.band_low_spin = ttk.Spinbox(left, from_=1, to=15, textvariable=self.band_low_var, width=8)
        self.band_low_spin.grid(row=r, column=3, sticky=tk.W, padx=5, pady=3)
        r += 1

        ttk.Label(left, text="Detect MAD k:").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.detect_k_var = tk.DoubleVar(value=float(self.declicker.default_params["detect_mad_k"]))
        ttk.Entry(left, textvariable=self.detect_k_var, width=10)\
            .grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)
        r += 1

        ttk.Separator(left, orient=tk.HORIZONTAL).grid(row=r, column=0, columnspan=4, sticky=tk.EW, pady=8)
        r += 1

        ttk.Label(left, text="Event (ms):").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.event_ms_var = tk.DoubleVar(value=float(self.declicker.default_params["event_ms"]))
        ttk.Entry(left, textvariable=self.event_ms_var, width=10)\
            .grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)

        ttk.Label(left, text="Context (ms):").grid(row=r, column=2, sticky=tk.W, pady=3)
        self.context_ms_var = tk.DoubleVar(value=float(self.declicker.default_params["context_ms"]))
        ttk.Entry(left, textvariable=self.context_ms_var, width=10)\
            .grid(row=r, column=3, sticky=tk.W, padx=5, pady=3)
        r += 1

        ttk.Label(left, text="Sigma trim q:").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.sigma_trim_q_var = tk.DoubleVar(value=float(self.declicker.default_params["sigma_trim_q"]))
        ttk.Entry(left, textvariable=self.sigma_trim_q_var, width=10)\
            .grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)
        r += 1

        ttk.Separator(left, orient=tk.HORIZONTAL).grid(row=r, column=0, columnspan=4, sticky=tk.EW, pady=8)
        r += 1

        ttk.Label(left, text="Reaction (0..1):").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.reaction_var = tk.DoubleVar(value=float(self.declicker.default_params["reaction"]))
        ttk.Scale(left, from_=0.0, to=1.0, variable=self.reaction_var, orient=tk.HORIZONTAL, length=180)\
            .grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)
        ttk.Label(left, textvariable=self.reaction_var).grid(row=r, column=2, sticky=tk.W, pady=3)
        r += 1

        ttk.Label(left, text="k_min:").grid(row=r, column=0, sticky=tk.W, pady=3)
        self.k_min_var = tk.DoubleVar(value=float(self.declicker.default_params["k_min"]))
        ttk.Entry(left, textvariable=self.k_min_var, width=10)\
            .grid(row=r, column=1, sticky=tk.W, padx=5, pady=3)

        ttk.Label(left, text="k_max:").grid(row=r, column=2, sticky=tk.W, pady=3)
        self.k_max_var = tk.DoubleVar(value=float(self.declicker.default_params["k_max"]))
        ttk.Entry(left, textvariable=self.k_max_var, width=10)\
            .grid(row=r, column=3, sticky=tk.W, padx=5, pady=3)
        r += 1

        self.info_text = tk.Text(left, height=14, width=70)
        self.info_text.grid(row=r, column=0, columnspan=4, sticky=tk.W + tk.E, pady=10)
        r += 1

        # Plot:
        self.fig = Figure(figsize=(9, 8.3), dpi=100)
        self.canvas = FigureCanvasTkAgg(self.fig, master=right)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

        self.status_var = tk.StringVar(value="Готов к работе")
        status = ttk.Label(self.root, textvariable=self.status_var, relief=tk.SUNKEN, anchor=tk.W)
        status.pack(side=tk.BOTTOM, fill=tk.X)

        self.level_var.trace_add("write", self._on_level_change)
        self._apply_level_to_band_spinboxes()

    def _update_wavelet_list(self, *args):
        fam = self.wavelet_family_var.get()
        wlist = self.declicker.wavelet_families.get(fam, ["db4"])
        self.wavelet_combo["values"] = wlist
        if self.wavelet_var.get() not in wlist:
            self.wavelet_var.set(wlist[0])

    def _apply_level_to_band_spinboxes(self):
        lvl = int(self.level_var.get())
        lvl = max(1, min(lvl, 15))
        self.band_high_spin.config(to=lvl)
        self.band_low_spin.config(to=lvl)

        bh = int(self.band_high_var.get())
        bl = int(self.band_low_var.get())
        bh = max(1, min(bh, lvl))
        bl = max(1, min(bl, lvl))
        self.band_high_var.set(bh)
        self.band_low_var.set(bl)

    def _on_level_change(self, *args):
        self._apply_level_to_band_spinboxes()
        if self.audio_loaded and self.declicker.sample_rate:
            self._write_info_with_band_table()

    def _collect_params(self):
        return {
            "wavelet": self.wavelet_var.get(),
            "border_mode": self.border_mode_var.get().strip(),
            "level": int(self.level_var.get()),
            "cycle_spins": int(self.cycle_spins_var.get()),

            "band_high": int(self.band_high_var.get()),
            "band_low": int(self.band_low_var.get()),

            "detect_mad_k": float(self.detect_k_var.get()),

            "event_ms": float(self.event_ms_var.get()),
            "context_ms": float(self.context_ms_var.get()),
            "sigma_trim_q": float(self.sigma_trim_q_var.get()),

            "reaction": float(self.reaction_var.get()),
            "k_min": float(self.k_min_var.get()),
            "k_max": float(self.k_max_var.get()),
        }

    def _write_info_with_band_table(self):
        sr = int(self.declicker.sample_rate)
        x = np.asarray(self.declicker.audio_data, dtype=np.float64)
        lvl = int(self.level_var.get())
        lvl = max(1, min(lvl, 15))

        info = []
        if self.loaded_path:
            info.append(f"Файл: {os.path.basename(self.loaded_path)}")
        info.append(f"Fs: {sr} Hz")
        info.append(f"Длина: {len(x)/sr:.2f} сек")
        info.append(f"Каналы: {'Стерео' if self.declicker.is_stereo else 'Моно'}")
        info.append(f"Max|x| (mono view): {np.max(np.abs(x)):.4f}")
        info.append("")
        info.append("Полосы DWT: cA_L (аппрокс.) + detail bands cD1..cD_L")
        fl0, fh0 = self.declicker.approx_hz(lvl)
        info.append(f"  cA{lvl}: ~ {fl0:.0f} .. {fh0:.0f} Hz")
        for j in range(lvl, 0, -1):
            fl, fh = self.declicker.detail_hz(j)
            info.append(f"  Band {j} (cD{j}): ~ {fl:.0f} .. {fh:.0f} Hz")

        self.info_text.delete(1.0, tk.END)
        self.info_text.insert(1.0, "\n".join(info))

    # plotting helpers

    @staticmethod
    def _downsample_for_display(x, max_points=200000):
        x = np.asarray(x)
        n = x.size
        if n <= max_points:
            return x, 1
        step = int(np.ceil(n / max_points))
        return x[::step], step

    def _show_loaded_preview_fragment(self):
        self.fig.clear()
        ax = self.fig.add_subplot(111)

        x = np.asarray(self.declicker.audio_data, dtype=np.float64)
        sr = float(self.declicker.sample_rate)

        display_len = min(40000, len(x))
        seg = x[:display_len]
        t = np.arange(display_len) / sr

        ax.plot(t, seg, linewidth=0.6, alpha=0.85)
        ax.set_title("Оригинал (исходный фрагмент)")
        ax.grid(True, alpha=0.3)
        ax.set_xlabel("Время (сек)")
        ax.set_ylabel("Амплитуда")
        self.fig.tight_layout()
        self.canvas.draw()

    def _show_full_original_and_cleaned(self):
        self.fig.clear()

        ax1 = self.fig.add_subplot(311)
        ax2 = self.fig.add_subplot(312)
        ax3 = self.fig.add_subplot(313)

        x = np.asarray(self.declicker.audio_data, dtype=np.float64)
        y = np.asarray(self.declicker.cleaned_audio, dtype=np.float64)
        sr = float(self.declicker.sample_rate)
        dur = len(x) / sr

        x_ds, step = self._downsample_for_display(x, max_points=250000)
        y_ds = y[::step]
        t_ds = (np.arange(len(x_ds)) * step) / sr

        # ---------- 1. Исходный сигнал ----------
        ax1.plot(t_ds, x_ds, linewidth=0.5, alpha=0.85)
        ax1.set_title("Исходный сигнал (полностью)")
        ax1.set_ylabel("Амплитуда")
        ax1.set_xlim(0, dur)
        ax1.grid(True, alpha=0.3)

        # ---------- 2. Очищенный сигнал ----------
        ax2.plot(t_ds, y_ds, linewidth=0.5, alpha=0.85)
        ax2.set_title("Очищенный сигнал (полностью)")
        ax2.set_ylabel("Амплитуда")
        ax2.set_xlim(0, dur)
        ax2.grid(True, alpha=0.3)

        # ---------- 3. Спектрограмма разности ----------
        diff = x - y

        nperseg = 1024
        noverlap = nperseg // 2

        if len(diff) >= nperseg:
            f, tt, Sxx = signal.spectrogram(
                diff,
                fs=sr,
                nperseg=nperseg,
                noverlap=noverlap
            )

            if tt.size > 0:
                tt = tt - tt[0]

            ax3.pcolormesh(
                tt,
                f,
                10 * np.log10(Sxx + 1e-10),
                shading="gouraud",
                cmap="hot"
            )

            ax3.set_title("Спектрограмма разности: original - cleaned")
            ax3.set_xlabel("Время (сек)")
            ax3.set_ylabel("Частота")
            ax3.set_xlim(0, dur)
            ax3.set_yscale("log")
            ax3.set_ylim(20, sr / 2)

            yticks = [100, 1000, 5000, 10000, 20000]
            yticks = [v for v in yticks if 20 <= v <= sr / 2]
            ax3.set_yticks(yticks)

            ylabels = []
            for v in yticks:
                if v >= 1000:
                    ylabels.append(f"{int(v / 1000)}k")
                else:
                    ylabels.append(f"{int(v)}")

            ax3.set_yticklabels(ylabels)
        else:
            ax3.text(
                0.5,
                0.5,
                "Файл слишком короткий для построения спектрограммы",
                ha="center",
                va="center",
                transform=ax3.transAxes
            )
            ax3.set_axis_off()

        # ---------- Общая временная сетка ----------
        max_sec = int(np.ceil(dur))

        if max_sec <= 20:
            tick_step = 1
        elif max_sec <= 60:
            tick_step = 5
        elif max_sec <= 180:
            tick_step = 10
        else:
            tick_step = 30

        xticks = list(range(0, max_sec + 1, tick_step))

        ax1.set_xticks(xticks)
        ax2.set_xticks(xticks)
        if len(diff) >= nperseg:
            ax3.set_xticks(xticks)

        self.fig.tight_layout()
        self.canvas.draw()

    # actions

    def load_audio(self):
        filepath = filedialog.askopenfilename(
            title="Выберите аудиофайл",
            filetypes=[("Audio files", "*.wav *.flac *.ogg"), ("All files", "*.*")]
        )
        if not filepath:
            return

        self.status_var.set(f"Загрузка: {os.path.basename(filepath)}...")
        self.root.update()

        ok = self.declicker.load_audio(filepath)
        if not ok:
            messagebox.showerror("Ошибка", "Не удалось загрузить аудиофайл")
            self.status_var.set("Ошибка загрузки")
            return

        self.loaded_path = filepath
        self.audio_loaded = True
        self.status_var.set(f"Загружен: {os.path.basename(filepath)}")

        self._write_info_with_band_table()
        self._show_loaded_preview_fragment()

    def process_audio(self):
        if not self.audio_loaded:
            messagebox.showwarning("Внимание", "Сначала загрузите аудио")
            return
        if self.processing:
            return

        params = self._collect_params()
        self.processing = True
        self.status_var.set("Обработка...")
        self.root.update()

        def worker():
            try:
                cleaned, clicks_n = self.declicker.apply_declicking(params)
                snr = self.declicker.calculate_snr(self.declicker.audio_data, self.declicker.cleaned_audio)
                self.root.after(0, lambda: self._after_processing(params, clicks_n, snr))
            except Exception as e:
                self.root.after(0, lambda: self._show_error(f"Ошибка обработки: {str(e)}"))
            finally:
                self.processing = False

        threading.Thread(target=worker, daemon=True).start()

    def _after_processing(self, params, clicks_n, snr):
        bh = params["band_high"]
        bl = params["band_low"]
        self.status_var.set(f"Готово. Кликов: {clicks_n}, SMR: {snr:.2f} dB (Bands {bh}..{bl})")

        # Append summary to info
        lines = self.info_text.get(1.0, tk.END).strip().splitlines()
        lines.append("")
        lines.append("--- Результаты ---")
        lines.append(f"Clicks: {clicks_n}")
        lines.append(f"SMR: {snr:.2f} dB")
        lines.append(f"Band range: {bh}..{bl} (inclusive)")
        lines.append(f"Event: {params['event_ms']} ms, Context: {params['context_ms']} ms, Cluster: {max(1.0, params['event_ms'])} ms")
        lines.append(f"Detect MAD k: {params['detect_mad_k']}")
        lines.append(f"Auto-k: reaction={params['reaction']}, k=[{params['k_min']}..{params['k_max']}]")
        self.info_text.delete(1.0, tk.END)
        self.info_text.insert(1.0, "\n".join(lines))

        # Main page: show full original + full cleaned
        self._show_full_original_and_cleaned()

    def save_result(self):
        if self.declicker.cleaned_audio is None and self.declicker.cleaned_stereo is None:
            messagebox.showwarning("Внимание", "Сначала обработайте аудио")
            return

        filepath = filedialog.asksaveasfilename(
            title="Сохранить результат",
            defaultextension=".wav",
            filetypes=[("WAV files", "*.wav"), ("All files", "*.*")]
        )
        if not filepath:
            return

        ok = self.declicker.save_cleaned_audio(filepath)
        if ok:
            self.status_var.set(f"Сохранено: {os.path.basename(filepath)}")
            messagebox.showinfo("Успех", "Файл сохранён")
        else:
            self.status_var.set("Ошибка сохранения")
            messagebox.showerror("Ошибка", "Не удалось сохранить файл")

    def visualize(self):
        if self.declicker.audio_data is None or self.declicker.cleaned_audio is None:
            messagebox.showwarning("Внимание", "Сначала обработайте аудио")
            return

        if not self.declicker.last_click_centers:
            messagebox.showinfo(
                "Нет кликов",
                "После обработки нет сохранённых позиций обнаруженных кликов."
            )
            return

        win = tk.Toplevel(self.root)
        win.title("Визуализация обнаруженных кликов")
        win.geometry("1200x900")

        control_frame = ttk.Frame(win)
        control_frame.pack(side=tk.TOP, fill=tk.X, padx=8, pady=6)

        ttk.Label(control_frame, text="Клик:").pack(side=tk.LEFT, padx=4)

        click_values = [
            f"{i + 1}: {center / self.declicker.sample_rate:.4f} сек"
            for i, center in enumerate(self.declicker.last_click_centers)
        ]

        selected_click = tk.IntVar(value=0)
        selected_label = tk.StringVar(value=click_values[0])

        click_combo = ttk.Combobox(
            control_frame,
            textvariable=selected_label,
            values=click_values,
            state="readonly",
            width=28
        )
        click_combo.pack(side=tk.LEFT, padx=4)

        ttk.Label(control_frame, text="Окно (мс):").pack(side=tk.LEFT, padx=(15, 4))

        window_ms_var = tk.DoubleVar(value=120.0)
        ttk.Entry(control_frame, textvariable=window_ms_var, width=8).pack(side=tk.LEFT, padx=4)

        fig_holder = {"fig": None}
        canvas_holder = {"canvas": None}

        plot_frame = ttk.Frame(win)
        plot_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        def get_selected_index():
            label = selected_label.get()
            try:
                return int(label.split(":")[0]) - 1
            except Exception:
                return int(selected_click.get())

        def redraw():
            idx = get_selected_index()
            idx = max(0, min(idx, len(self.declicker.last_click_centers) - 1))
            selected_click.set(idx)

            try:
                window_ms = float(window_ms_var.get())
            except Exception:
                window_ms = 120.0
                window_ms_var.set(window_ms)

            window_ms = max(10.0, min(window_ms, 2000.0))

            if canvas_holder["canvas"] is not None:
                canvas_holder["canvas"].get_tk_widget().destroy()

            if fig_holder["fig"] is not None:
                plt.close(fig_holder["fig"])

            fig = self.declicker.visualize_click_segment(
                click_index=idx,
                window_ms=window_ms
            )

            fig_holder["fig"] = fig

            canvas = FigureCanvasTkAgg(fig, master=plot_frame)
            canvas.draw()
            canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

            canvas_holder["canvas"] = canvas

        def go_prev():
            idx = get_selected_index()
            idx = max(0, idx - 1)
            selected_label.set(click_values[idx])
            selected_click.set(idx)
            redraw()

        def go_next():
            idx = get_selected_index()
            idx = min(len(click_values) - 1, idx + 1)
            selected_label.set(click_values[idx])
            selected_click.set(idx)
            redraw()

        ttk.Button(control_frame, text="← Назад", command=go_prev).pack(side=tk.LEFT, padx=8)
        ttk.Button(control_frame, text="Вперёд →", command=go_next).pack(side=tk.LEFT, padx=4)
        ttk.Button(control_frame, text="Обновить окно", command=redraw).pack(side=tk.LEFT, padx=8)
        ttk.Button(control_frame, text="Закрыть", command=win.destroy).pack(side=tk.RIGHT, padx=4)

        click_combo.bind("<<ComboboxSelected>>", lambda event: redraw())

        redraw()

    def _show_error(self, msg):
        messagebox.showerror("Ошибка", msg)
        self.status_var.set("Ошибка")

### Запуск программы

In [22]:
def main():
    root = tk.Tk()
    app = DeclickerGUI(root)
    root.mainloop()


if __name__ == "__main__":
    main()